In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor
from PIL import Image
from activation_store import ProbingDataset
from torch.utils.data import DataLoader
import torch
import torch.multiprocessing as mp
from tqdm import tqdm
import threading
from queue import Queue

import os
import matplotlib.pyplot as plt
import numpy as np
import json

# Load SAEs and hooked model for all 12 layers
from dictionary_learning.utils import load_dictionary
from hooked_model import HookedModel

In [ ]:
device = 'cuda:7'  # Use one GPU for SAE inference

# Load SAEs for CLS tokens (existing code)
saes_cls = {}
configs_cls = {}
for layer in range(12):
    dir = f"/Checkpoints/SAE/Layer-wise/CLS/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0"
    sae, config = load_dictionary(dir, device)
    saes_cls[layer] = sae
    configs_cls[layer] = config
    print(f"CLS Layer {layer}: k={config['trainer']['k']}, dict_size={config['trainer']['dict_size']}")

# Load SAEs for Image tokens (you'll need to update the path)
saes_img = {}
configs_img = {}
for layer in range(12):
    dir = f"/Checkpoints/SAE/Layer-wise/Image/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0"  
    sae, config = load_dictionary(dir, device)
    saes_img[layer] = sae
    configs_img[layer] = config
    print(f"Image Layer {layer}: k={config['trainer']['k']}, dict_size={config['trainer']['dict_size']}")

# Load hooked model with all layer hook points
hook_points = [f"layer-{i}_resid-post" for i in range(12)]
hooked_model = HookedModel(
    "openai/clip-vit-base-patch32", 
    hook_points=hook_points, 
    device=device
)

In [ ]:
def load_image_from_path(image_path):
    """Load and display an image from an arbitrary file path"""
    try:
        # Check if file exists
        if not os.path.exists(image_path):
            print(f"Error: File not found at {image_path}")
            return None
        
        # Check if it's a valid image file
        valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp', '.gif']
        file_ext = os.path.splitext(image_path)[1].lower()
        if file_ext not in valid_extensions:
            print(f"Error: Unsupported file format {file_ext}")
            print(f"Supported formats: {', '.join(valid_extensions)}")
            return None
        
        # Load and convert image
        image = Image.open(image_path).convert('RGB')
        
        # Display image info
        filename = os.path.basename(image_path)
        print(f"Successfully loaded image: {filename}")
        print(f"Image size: {image.size}")
        print(f"Image mode: {image.mode}")
        
        # Display the image
        plt.figure(figsize=(8, 8))
        plt.imshow(image)
        # plt.title(f"Loaded image: {filename}")
        plt.axis('off')
        plt.show()
        
        return image
        
    except Exception as e:
        print(f"Error loading image from {image_path}: {str(e)}")
        return None


image_path = "/images/cat_and_dog.jpg"



image = load_image_from_path(image_path)

In [ ]:
# # Get activations using the hooked model's method
# # Pass the raw PIL image, not the preprocessed tensor
# embeddings = hooked_model.get_activations_from_image(image)

# print(f"Got activations for {len(embeddings)} layers")
# for layer_name, act in embeddings.items():
#     print(f"{layer_name}: {act.shape}")


# Get activations using the hooked model's method
# Pass the raw PIL image, not the preprocessed tensor
embeddings = hooked_model.get_activations_from_image(image)

# # Normalize embeddings to match training setup
# normalized_embeddings = {}
# for layer_name, act in embeddings.items():
#     # Apply L2 normalization along the last dimension (feature dimension)
#     normalized_embeddings[layer_name] = act / act.norm(dim=-1, keepdim=True)

# # Replace original embeddings with normalized ones
# embeddings = normalized_embeddings

print(f"Got normalized activations for {len(embeddings)} layers")
for layer_name, act in embeddings.items():
    print(f"{layer_name}: {act.shape}")

In [ ]:
# Load frequent feature indices for each layer
freq_idx_cls = {}
freq_idx_img = {}

for layer in range(12):
    cls_path = f"/Checkpoints/SAE/Layer-wise/CLS/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0/freq_feat_idx.pt"
    img_path = f"/Checkpoints/SAE/Layer-wise/Image/BatchTopK-orig_top-128/ef_8/layer_{layer}/trainer_0/freq_feat_idx.pt"
    freq_idx_cls[layer] = torch.load(cls_path).cpu().numpy()
    freq_idx_img[layer] = torch.load(img_path).cpu().numpy()

In [ ]:
# Load concept files for each layer
concept_files_path_cls = "/dissect/CLS/BatchTopK-orig_top-128/ef-8/top1_concepts_gpt5-all"
concept_files_path_img = "/dissect/Image/BatchTopK-orig_top-128/ef-8/top1_concepts_meanpooled_gpt5-all"

layer_concepts_cls = {}
layer_concepts_img = {}

print("Loading concept files for CLS and Image tokens...")
for layer in range(12):
    # CLS concepts
    concept_file_cls = f"{concept_files_path_cls}/layer_{layer}_top1_concepts.txt"
    try:
        with open(concept_file_cls, 'r') as f:
            concepts_cls = [line.strip() for line in f.readlines()]
        layer_concepts_cls[layer] = concepts_cls
        print(f"CLS Layer {layer}: Loaded {len(concepts_cls)} concepts")
    except FileNotFoundError:
        print(f"Warning: CLS concept file not found for layer {layer}")
        layer_concepts_cls[layer] = []

    # Image concepts
    concept_file_img = f"{concept_files_path_img}/layer_{layer}_top1_concepts.txt"
    try:
        with open(concept_file_img, 'r') as f:
            concepts_img = [line.strip() for line in f.readlines()]
        layer_concepts_img[layer] = concepts_img
        print(f"Image Layer {layer}: Loaded {len(concepts_img)} concepts")
    except FileNotFoundError:
        print(f"Warning: Image concept file not found for layer {layer}")
        layer_concepts_img[layer] = []

print("Done loading concept files for CLS and Image tokens!")

In [ ]:
def get_top_k_concepts_unified(
    sae, 
    activation, 
    freq_idx, 
    concept_list, 
    top_k=5, 
    token_type="cls"
):
    """
    Unified function to get top-k activated concepts with their indices, values, and names.
    
    Args:
        sae: SAE model for encoding
        activation: Tensor activation(s) - for CLS: [hidden_dim], for IMG: [num_tokens, hidden_dim]
        freq_idx: Frequent feature indices to mask out
        concept_list: List of concept names for this layer
        top_k: Number of top concepts to return
        token_type: "cls" or "img" - determines how to handle the activation tensor
    
    Returns:
        dict with:
            'indices': np.array of feature indices
            'values': np.array of activation values
            'concepts': list of concept names
            'valid_mask': boolean mask for non-zero activations
    """
    with torch.no_grad():
        if token_type.lower() == "cls":
            # CLS token: activation should be [hidden_dim]
            if activation.dim() > 1:
                activation = activation.squeeze()
            feature_acts = sae.encode(activation.unsqueeze(0)).squeeze(0)
        elif token_type.lower() == "img":
            # IMG tokens: activation should be [num_tokens, hidden_dim]
            if activation.dim() == 1:
                activation = activation.unsqueeze(0)
            feature_acts = sae.encode(activation)  # [num_tokens, dict_size]
            feature_acts = feature_acts.mean(dim=0)  # Average across tokens -> [dict_size]
        else:
            raise ValueError(f"token_type must be 'cls' or 'img', got {token_type}")
        
        # Zero out frequent features
        feature_acts[freq_idx] = 0
        
        # Get top-k values and indices
        top_values, top_indices = torch.topk(feature_acts, top_k)
        
        # Filter out zero activations
        nonzero_mask = top_values != 0
        valid_indices = top_indices[nonzero_mask].cpu().numpy()
        valid_values = top_values[nonzero_mask].cpu().numpy()
        
        # Get concept names for valid indices
        concepts = []
        for idx in valid_indices:
            if idx < len(concept_list):
                concepts.append(concept_list[idx])
            else:
                concepts.append(f"feat_{idx}")
        
        return {
            'indices': valid_indices,
            'values': valid_values, 
            'concepts': concepts,
            'valid_mask': nonzero_mask.cpu().numpy(),
            'raw_feature_acts': feature_acts  # Include for advanced use cases
        }

def get_top_k_concepts_all_layers(
    embeddings, 
    saes_cls, 
    saes_img, 
    freq_idx_cls, 
    freq_idx_img,
    layer_concepts_cls, 
    layer_concepts_img, 
    top_k=5,
    layers=None
):
    """
    Get top-k concepts for all specified layers at once.
    
    Args:
        embeddings: Dict of layer activations from hooked model
        saes_cls, saes_img: Dict of SAE models by layer
        freq_idx_cls, freq_idx_img: Dict of frequent indices by layer
        layer_concepts_cls, layer_concepts_img: Dict of concept lists by layer
        top_k: Number of top concepts per layer
        layers: List of layers to process (default: 0-11)
    
    Returns:
        dict with structure: {layer: {'CLS': result_dict, 'IMG': result_dict}}
    """
    if layers is None:
        layers = range(12)
    
    all_results = {}
    
    for layer in layers:
        layer_name = f"layer-{layer}_resid-post"
        layer_activations = embeddings[layer_name]
        
        all_results[layer] = {}
        
        # CLS concepts
        cls_activation = layer_activations[0, 0, :]
        cls_result = get_top_k_concepts_unified(
            sae=saes_cls[layer],
            activation=cls_activation,
            freq_idx=freq_idx_cls[layer],
            concept_list=layer_concepts_cls.get(layer, []),
            top_k=top_k,
            token_type="cls"
        )
        all_results[layer]['CLS'] = cls_result
        
        # IMG concepts
        img_activations = layer_activations[0, 1:, :]
        img_result = get_top_k_concepts_unified(
            sae=saes_img[layer],
            activation=img_activations,
            freq_idx=freq_idx_img[layer],
            concept_list=layer_concepts_img.get(layer, []),
            top_k=top_k,
            token_type="img"
        )
        all_results[layer]['IMG'] = img_result
    
    return all_results

In [ ]:
all_concepts = get_top_k_concepts_all_layers(
    embeddings=embeddings,  # Your image embeddings from hooked model
    saes_cls=saes_cls,      # Dictionary of CLS SAE models by layer
    saes_img=saes_img,      # Dictionary of IMG SAE models by layer
    freq_idx_cls=freq_idx_cls,  # Dictionary of frequent indices to mask out
    freq_idx_img=freq_idx_img,
    layer_concepts_cls=layer_concepts_cls,  # Dictionary of concept lists by layer
    layer_concepts_img=layer_concepts_img,
    top_k=10  # Get top 5 concepts per layer per token type
)

In [ ]:
for layer in range(12):
    print(f"\n=== Layer {layer} ===")
    
    # CLS concepts for this layer
    cls_data = all_concepts[layer]['CLS']
    print(f"CLS concepts: {cls_data['concepts']}")
    print(f"CLS indices: {cls_data['indices']}")
    print(f"CLS values: {cls_data['values']}")
    
    # IMG concepts for this layer  
    img_data = all_concepts[layer]['IMG']
    print(f"IMG concepts: {img_data['concepts']}")
    print(f"IMG indices: {img_data['indices']}")
    print(f"IMG values: {img_data['values']}")

In [ ]:
def causal_tracing_cls_general(
    image, layer_L, layer_Lp, saes_cls, saes_img, hooked_model, freq_idx_cls, 
    layer_concepts_cls, layer_concepts_img, all_concepts=None, top_k=10, edit_factor=0
):
    """
    Returns: normalized_influence_cls2cls [top_k, num_cls_Lp], normalized_influence_cls2img [top_k, num_img_Lp], concept lists
    
    Args:
        all_concepts: Pre-computed concepts from get_top_k_concepts_all_layers (optional)
        Other args same as before
    """
    embeddings = hooked_model.get_activations_from_image(image)
    layer_name_L = f"layer-{layer_L}_resid-post"
    layer_name_Lp = f"layer-{layer_Lp}_resid-post"

    # CLS activation at source layer
    cls_activation_L = embeddings[layer_name_L][0, 0, :]
    sae_L = saes_cls[layer_L]
    
    # Get top features for source layer - use all_concepts if available
    if all_concepts is not None:
        # Use pre-computed concepts and indices
        cls_data_L = all_concepts[layer_L]['CLS']
        top_indices_L = cls_data_L['indices']
        # Get raw feature activations for modification
        feature_acts_L = sae_L.encode(cls_activation_L.unsqueeze(0), return_raw=True).squeeze(0)
    else:
        # Original logic as fallback
        feature_acts_L = sae_L.encode(cls_activation_L.unsqueeze(0), return_raw=True).squeeze(0)
        feature_acts_L_freq = feature_acts_L.clone()
        feature_acts_L_freq[freq_idx_cls[layer_L]] = 0
        top_values, top_indices = torch.topk(feature_acts_L_freq, top_k)
        nonzero_mask = top_values != 0
        top_indices_L = top_indices[nonzero_mask].cpu().numpy()

    # Get SAE for target layer
    sae_Lp_cls = saes_cls[layer_Lp]
    sae_Lp_img = saes_img[layer_Lp]

    # Get activations at target layer
    cls_activation_Lp = embeddings[layer_name_Lp][0, 0, :]
    img_activations_Lp = embeddings[layer_name_Lp][0, 1:, :]

    feature_acts_Lp_cls_orig = sae_Lp_cls.encode(cls_activation_Lp.unsqueeze(0), return_raw=True).squeeze(0)
    with torch.no_grad():
        img_sae_acts_orig = sae_Lp_img.encode(img_activations_Lp, return_raw=True)
        feature_acts_Lp_img_orig = img_sae_acts_orig.mean(dim=0)

    num_cls_Lp = feature_acts_Lp_cls_orig.shape[0]
    num_img_Lp = feature_acts_Lp_img_orig.shape[0]
    normalized_influence_cls2cls = np.zeros((len(top_indices_L), num_cls_Lp))
    normalized_influence_cls2img = np.zeros((len(top_indices_L), num_img_Lp))

    eps = 1e-8

    # For each top feature in source layer, ablate and propagate to target layer
    for i, idx_L in enumerate(top_indices_L):
        feature_acts_L_mod = feature_acts_L.clone()
        feature_acts_L_mod[idx_L] = edit_factor

        recon_L = sae_L.decode(feature_acts_L_mod.unsqueeze(0)).squeeze(0)
        layer_output_L = embeddings[layer_name_L].clone()
        layer_output_L[0, 0, :] = recon_L

        # Propagate from layer_L to layer_Lp
        x = layer_output_L
        for k in range(layer_L + 1, layer_Lp + 1):
            x = hooked_model.model.encoder.layers[k](x, attention_mask=None, causal_attention_mask=None)
            if isinstance(x, tuple):
                x = x[0]

        new_cls_activation_Lp = x[0, 0, :]
        new_img_activations_Lp = x[0, 1:, :]

        feature_acts_Lp_cls_new = sae_Lp_cls.encode(new_cls_activation_Lp.unsqueeze(0), return_raw=True).squeeze(0)
        with torch.no_grad():
            img_sae_acts = sae_Lp_img.encode(new_img_activations_Lp, return_raw=True)
            new_avg_img_sae_acts = img_sae_acts.mean(dim=0)
        feature_acts_Lp_img_new = new_avg_img_sae_acts

        # Influence on target-layer CLS concepts
        orig = feature_acts_Lp_cls_orig.detach().cpu().numpy()
        new = feature_acts_Lp_cls_new.detach().cpu().numpy()
        diff = orig - new
        diff = np.maximum(diff, 0)
        norm_diff = np.where(orig > eps, diff / (orig + eps), 0)
        normalized_influence_cls2cls[i, :] = norm_diff

        # Influence on target-layer IMG concepts
        orig_img = feature_acts_Lp_img_orig.detach().cpu().numpy()
        new_img = feature_acts_Lp_img_new.detach().cpu().numpy()
        diff_img = orig_img - new_img
        diff_img = np.maximum(diff_img, 0)
        norm_diff_img = np.where(orig_img > eps, diff_img / (orig_img + eps), 0)
        normalized_influence_cls2img[i, :] = norm_diff_img

    # Get concept names - use all_concepts if available
    if all_concepts is not None:
        concepts_L = all_concepts[layer_L]['CLS']['concepts']
    else:
        # Original logic as fallback
        concepts_L = [layer_concepts_cls[layer_L][idx] if idx < len(layer_concepts_cls[layer_L]) else f"feat_{idx}" for idx in top_indices_L]

    return normalized_influence_cls2cls, normalized_influence_cls2img, concepts_L

In [ ]:
# Usage with all_concepts
layer_L = 1
influence_cls2cls, influence_cls2img, concepts_L = causal_tracing_cls_general(
    image, layer_L, layer_L+9, saes_cls, saes_img, hooked_model, freq_idx_cls, 
    layer_concepts_cls, layer_concepts_img, all_concepts=all_concepts, top_k=5
)
print(influence_cls2cls.shape)  # [10, 6144] if dict_size=6144
print(influence_cls2cls.max())
print(influence_cls2img.max())
print(concepts_L)

In [ ]:
def causal_tracing_img_general(
    image, layer_L, layer_Lp, saes_cls, saes_img, hooked_model, freq_idx_img, 
    layer_concepts_cls, layer_concepts_img, all_concepts=None, top_k=10
):
    """
    Returns: normalized_influence_img2cls [top_k, num_cls_Lp], normalized_influence_img2img [top_k, num_img_Lp], concept lists
    
    Args:
        all_concepts: Pre-computed concepts from get_top_k_concepts_all_layers (optional)
        Other args same as before
    """
    embeddings = hooked_model.get_activations_from_image(image)
    layer_name_L = f"layer-{layer_L}_resid-post"
    layer_name_Lp = f"layer-{layer_Lp}_resid-post"

    img_activations_L = embeddings[layer_name_L][0, 1:, :]  # [num_img_tokens, hidden_dim]
    sae_L = saes_img[layer_L]
    
    # Get top features for source layer - use all_concepts if available
    if all_concepts is not None:
        # Use pre-computed concepts and indices
        img_data_L = all_concepts[layer_L]['IMG']
        top_indices_L = img_data_L['indices']
        # Get raw feature activations for modification
        with torch.no_grad():
            feature_acts_L = sae_L.encode(img_activations_L, return_raw=True)  # [num_img_tokens, dict_size]
    else:
        # Original logic as fallback
        with torch.no_grad():
            feature_acts_L = sae_L.encode(img_activations_L, return_raw=True)  # [num_img_tokens, dict_size]
            feature_acts_L[:, freq_idx_img[layer_L]] = 0
            avg_feature_acts_L = feature_acts_L.mean(dim=0)  # [dict_size]
            top_values, top_indices = torch.topk(avg_feature_acts_L, top_k)
            nonzero_mask = top_values != 0
            top_indices_L = top_indices[nonzero_mask].cpu().numpy()

    sae_Lp_cls = saes_cls[layer_Lp]
    sae_Lp_img = saes_img[layer_Lp]

    cls_activation_Lp = embeddings[layer_name_Lp][0, 0, :]
    img_activations_Lp = embeddings[layer_name_Lp][0, 1:, :]

    feature_acts_Lp_cls_orig = sae_Lp_cls.encode(cls_activation_Lp.unsqueeze(0), return_raw=True).squeeze(0)
    with torch.no_grad():
        img_sae_acts_orig = sae_Lp_img.encode(img_activations_Lp, return_raw=True)
        feature_acts_Lp_img_orig = img_sae_acts_orig.mean(dim=0)

    num_cls_Lp = feature_acts_Lp_cls_orig.shape[0]
    num_img_Lp = feature_acts_Lp_img_orig.shape[0]
    normalized_influence_img2cls = np.zeros((len(top_indices_L), num_cls_Lp))
    normalized_influence_img2img = np.zeros((len(top_indices_L), num_img_Lp))

    eps = 1e-8

    for i, idx_L in enumerate(top_indices_L):
        feature_acts_L_mod = feature_acts_L.clone()
        feature_acts_L_mod[:, idx_L] = 0  # Ablate this feature for all tokens

        recon_img_tokens = sae_L.decode(feature_acts_L_mod)  # [num_img_tokens, hidden_dim]

        layer_output_L = embeddings[layer_name_L].clone()
        layer_output_L[0, 1:, :] = recon_img_tokens  # Replace all image tokens

        # Propagate from layer_L to layer_Lp
        x = layer_output_L
        for k in range(layer_L + 1, layer_Lp + 1):
            x = hooked_model.model.encoder.layers[k](x, attention_mask=None, causal_attention_mask=None)
            if isinstance(x, tuple):
                x = x[0]

        new_cls_activation_Lp = x[0, 0, :]
        new_img_activations_Lp = x[0, 1:, :]

        feature_acts_Lp_cls_new = sae_Lp_cls.encode(new_cls_activation_Lp.unsqueeze(0), return_raw=True).squeeze(0)
        with torch.no_grad():
            img_sae_acts = sae_Lp_img.encode(new_img_activations_Lp, return_raw=True)
            new_avg_img_sae_acts = img_sae_acts.mean(dim=0)
        feature_acts_Lp_img_new = new_avg_img_sae_acts

        # Influence on target-layer CLS concepts
        orig = feature_acts_Lp_cls_orig.detach().cpu().numpy()
        new = feature_acts_Lp_cls_new.detach().cpu().numpy()
        diff = orig - new
        diff = np.maximum(diff, 0)
        norm_diff = np.where(orig > eps, diff / (orig + eps), 0)
        normalized_influence_img2cls[i, :] = norm_diff

        # Influence on target-layer IMG concepts
        orig_img = feature_acts_Lp_img_orig.detach().cpu().numpy()
        new_img = feature_acts_Lp_img_new.detach().cpu().numpy()
        diff_img = orig_img - new_img
        diff_img = np.maximum(diff_img, 0)
        norm_diff_img = np.where(orig_img > eps, diff_img / (orig_img + eps), 0)
        normalized_influence_img2img[i, :] = norm_diff_img

    # Get concept names - use all_concepts if available
    if all_concepts is not None:
        concepts_L = all_concepts[layer_L]['IMG']['concepts']
    else:
        # Original logic as fallback
        concepts_L = [layer_concepts_img[layer_L][idx] if idx < len(layer_concepts_img[layer_L]) else f"feat_{idx}" for idx in top_indices_L]

    return normalized_influence_img2cls, normalized_influence_img2img, concepts_L

In [ ]:
influence_img2cls, influence_img2img, concepts_L_img = causal_tracing_img_general(
    image, layer_L, layer_L+1, saes_cls, saes_img, hooked_model, freq_idx_img, 
    layer_concepts_cls, layer_concepts_img, all_concepts=all_concepts, top_k=5
)

print(influence_img2cls.shape)  # [10, 6144] if dict_size=6144
print(influence_img2cls.max())
print(influence_img2img.max())
print(concepts_L)

In [ ]:
top_k_global = 3

In [ ]:
# COMPLETE G_fixed CONSTRUCTION - properly map concept positions to feature indices for ALL edge types

import networkx as nx
import random

G = nx.DiGraph()
node_pos = {}
layer_gap = 4
cls_gap = 2.5
img_gap = 2.5
top_k = top_k_global  # number of top concepts to consider per layer per token type
cls_base_y = top_k * 3
img_base_y = 0

weight_threshold = 0.0

num_layers = 12

# Recreate the graph with proper indexing using sliced all_concepts
G_fixed = nx.DiGraph()
node_pos_fixed = {}

# Add nodes using sliced pre-computed concepts from all_concepts
for layer in range(num_layers):
    # CLS nodes - slice the first top_k concepts
    cls_data = all_concepts[layer]['CLS']
    concepts_cls = cls_data['concepts'][:top_k]  # Slice to get only first top_k
    
    for i, c in enumerate(concepts_cls):
        jitter = random.uniform(-0.3, 0.3)
        node = f"L{layer}:CLS:{c}"
        G_fixed.add_node(node)
        node_pos_fixed[node] = (layer * layer_gap, cls_base_y - i * cls_gap + jitter)
    
    # IMG nodes - slice the first top_k concepts
    img_data = all_concepts[layer]['IMG']
    concepts_img = img_data['concepts'][:top_k]  # Slice to get only first top_k
    
    for i, c in enumerate(concepts_img):
        jitter = random.uniform(-0.3, 0.3)
        node = f"L{layer}:IMG:{c}"
        G_fixed.add_node(node)
        node_pos_fixed[node] = (layer * layer_gap, img_base_y - i * img_gap + jitter)

print(f"Added nodes: {G_fixed.number_of_nodes()} total")

# Add edges with FIXED indexing for ALL edge types using sliced all_concepts
for layer_L in tqdm(range(num_layers)):
    for layer_Lp in range(layer_L + 1, num_layers):
        
        # Get sliced source concepts and their indices
        source_cls_data = all_concepts[layer_L]['CLS']
        source_img_data = all_concepts[layer_L]['IMG']
        concepts_cls_L = source_cls_data['concepts'][:top_k]  # Slice
        concepts_img_L = source_img_data['concepts'][:top_k]  # Slice
        
        # Get sliced target concepts and their indices
        target_cls_data = all_concepts[layer_Lp]['CLS']
        target_img_data = all_concepts[layer_Lp]['IMG']
        concepts_cls_Lp_top = target_cls_data['concepts'][:top_k]  # Slice
        target_cls_indices = target_cls_data['indices'][:top_k]    # Slice indices too!
        concepts_img_Lp_top = target_img_data['concepts'][:top_k]  # Slice
        target_img_indices = target_img_data['indices'][:top_k]    # Slice indices too!
        
        # 1. CLS->CLS with proper target index mapping
        influence_cls2cls, _, _ = causal_tracing_cls_general(
            image, layer_L, layer_Lp, saes_cls, saes_img, hooked_model, 
            freq_idx_cls, layer_concepts_cls, layer_concepts_img, 
            all_concepts=all_concepts, top_k=top_k
        )
        
        for i, c_L in enumerate(concepts_cls_L):
            for j, c_Lp in enumerate(concepts_cls_Lp_top):
                node_src = f"L{layer_L}:CLS:{c_L}"
                node_dst = f"L{layer_Lp}:CLS:{c_Lp}"
                
                # FIXED: Use sliced target_cls_indices[j] instead of j
                target_feature_idx = target_cls_indices[j]
                weight = influence_cls2cls[i, target_feature_idx] if i < influence_cls2cls.shape[0] and target_feature_idx < influence_cls2cls.shape[1] else 0
                
                if weight > weight_threshold and node_src in G_fixed.nodes() and node_dst in G_fixed.nodes():
                    G_fixed.add_edge(node_src, node_dst, weight=weight)

        # 2. CLS->IMG with proper target index mapping
        _, influence_cls2img, _ = causal_tracing_cls_general(
            image, layer_L, layer_Lp, saes_cls, saes_img, hooked_model, 
            freq_idx_cls, layer_concepts_cls, layer_concepts_img,
            all_concepts=all_concepts, top_k=top_k
        )
        
        for i, c_L in enumerate(concepts_cls_L):
            for j, c_Lp in enumerate(concepts_img_Lp_top):
                node_src = f"L{layer_L}:CLS:{c_L}"
                node_dst = f"L{layer_Lp}:IMG:{c_Lp}"
                
                # FIXED: Use sliced target_img_indices[j] instead of j
                target_feature_idx = target_img_indices[j]
                weight = influence_cls2img[i, target_feature_idx] if i < influence_cls2img.shape[0] and target_feature_idx < influence_cls2img.shape[1] else 0
                
                if weight > weight_threshold and node_src in G_fixed.nodes() and node_dst in G_fixed.nodes():
                    G_fixed.add_edge(node_src, node_dst, weight=weight)

        # 3. IMG->CLS with proper target index mapping
        influence_img2cls, _, _ = causal_tracing_img_general(
            image, layer_L, layer_Lp, saes_cls, saes_img, hooked_model, 
            freq_idx_img, layer_concepts_cls, layer_concepts_img,
            all_concepts=all_concepts, top_k=top_k
        )
        
        for i, c_L in enumerate(concepts_img_L):
            for j, c_Lp in enumerate(concepts_cls_Lp_top):
                node_src = f"L{layer_L}:IMG:{c_L}"
                node_dst = f"L{layer_Lp}:CLS:{c_Lp}"
                
                # FIXED: Use sliced target_cls_indices[j] instead of j
                target_feature_idx = target_cls_indices[j]
                weight = influence_img2cls[i, target_feature_idx] if i < influence_img2cls.shape[0] and target_feature_idx < influence_img2cls.shape[1] else 0
                
                if weight > weight_threshold and node_src in G_fixed.nodes() and node_dst in G_fixed.nodes():
                    G_fixed.add_edge(node_src, node_dst, weight=weight)

        # 4. IMG->IMG with proper target index mapping
        _, influence_img2img, _ = causal_tracing_img_general(
            image, layer_L, layer_Lp, saes_cls, saes_img, hooked_model, 
            freq_idx_img, layer_concepts_cls, layer_concepts_img,
            all_concepts=all_concepts, top_k=top_k
        )
        
        for i, c_L in enumerate(concepts_img_L):
            for j, c_Lp in enumerate(concepts_img_Lp_top):
                node_src = f"L{layer_L}:IMG:{c_L}"
                node_dst = f"L{layer_Lp}:IMG:{c_Lp}"
                
                # FIXED: Use sliced target_img_indices[j] instead of j
                target_feature_idx = target_img_indices[j]
                weight = influence_img2img[i, target_feature_idx] if i < influence_img2img.shape[0] and target_feature_idx < influence_img2img.shape[1] else 0
                
                if weight > weight_threshold and node_src in G_fixed.nodes() and node_dst in G_fixed.nodes():
                    G_fixed.add_edge(node_src, node_dst, weight=weight)

print(f"COMPLETE Fixed graph: {G_fixed.number_of_nodes()} nodes, {G_fixed.number_of_edges()} edges")

# Check if L11 connections now exist
l11_incoming_fixed = [(u, v) for u, v in G_fixed.edges() if v.startswith("L11:")]
print(f"L11 incoming edges in COMPLETE fixed graph: {len(l11_incoming_fixed)}")
for u, v in l11_incoming_fixed[:10]:
    print(f"  {u} → {v}")

# Compare with original buggy graph (if it exists)
if 'G' in locals():
    print(f"\nComparison:")
    print(f"  Original G:      {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"  Fixed G_fixed:   {G_fixed.number_of_nodes()} nodes, {G_fixed.number_of_edges()} edges")
    print(f"  Edge difference: {G_fixed.number_of_edges() - G.number_of_edges()} edges")

# Verify the fix worked by checking some edge weights
print(f"\nVerifying edge weight consistency:")
sample_edges = list(G_fixed.edges(data=True))[:5]
for u, v, d in sample_edges:
    fixed_weight = d['weight']
    print(f"  {u} → {v}: Weight={fixed_weight:.6f}")

In [ ]:
import os

concept_files = {
    1: ["color.txt", "edge.txt"],
    2: ["texture.txt", "shape.txt"],
    3: ["object.txt", "part.txt"],
    4: ["motion.txt", "relation.txt"],
}
concept_level_map = {}
concept_dir = "/concept_set"

for level, files in concept_files.items():
    for fname in files:
        with open(os.path.join(concept_dir, fname), "r") as f:
            for line in f:
                concept = line.strip()
                if concept:  # skip empty lines
                    concept_level_map[concept] = level

# Define colors for each level
level_colors = {
    1: "#1f77b4",  # blue
    2: "#2ca02c",  # green
    3: "#ff7f0e",  # orange
    4: "#d62728",  # red
}

In [ ]:
# Plot the original G_fixed graph using matplotlib
import matplotlib.pyplot as plt
import networkx as nx
import random

# Set random seed for consistent layout
random.seed(42)

# Use the existing node positions from node_pos_fixed
plt.figure(figsize=(layer_gap * (num_layers + 1), 16))

# Node colors based on concept level (same as before)
node_colors_fixed = []
for node in G_fixed.nodes():
    concept = node.split(":")[-1]
    level = concept_level_map.get(concept, None)
    color = level_colors.get(level, "#cccccc")
    node_colors_fixed.append(color)

# Node sizes - uniform for all nodes
node_sizes_fixed = [800 for _ in G_fixed.nodes()]

# Draw nodes
nx.draw_networkx_nodes(
    G_fixed,
    pos=node_pos_fixed,
    node_color=node_colors_fixed,
    node_size=node_sizes_fixed,
    alpha=0.8,
    edgecolors='black',
    linewidths=1.0
)

# Node labels: show only concept name (not the full node identifier)
node_labels_fixed = {node: node.split(":")[-1] for node in G_fixed.nodes()}
nx.draw_networkx_labels(
    G_fixed,
    pos=node_pos_fixed,
    labels=node_labels_fixed,
    font_size=12,  # Smaller font since there are many nodes
    font_weight='bold',
    font_color='black'
)

# Draw edges with thickness proportional to weight
if G_fixed.number_of_edges() > 0:
    edges = list(G_fixed.edges(data=True))
    weights = [d['weight'] for (_, _, d) in edges]
    max_weight = max(weights) if weights else 1.0
    
    # Calculate edge widths proportional to weights
    edge_widths = [max(0.3, min(4, d['weight'] * 4 / max_weight)) for (_, _, d) in edges]
    
    # Draw edges
    nx.draw_networkx_edges(
        G_fixed,
        pos=node_pos_fixed,
        edgelist=[(u, v) for u, v, d in edges],
        width=edge_widths,
        edge_color='gray',
        arrows=True,
        arrowsize=15,
        alpha=0.6,
        arrowstyle='->'
    )
    
    # Optional: Add edge labels for strongest connections only
    # (Uncomment if you want to see edge weights)
    # strong_edges = [(u, v, d) for u, v, d in edges if d['weight'] > max_weight * 0.5]
    # edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in strong_edges}
    # nx.draw_networkx_edge_labels(
    #     G_fixed,
    #     pos=node_pos_fixed,
    #     edge_labels=edge_labels,
    #     font_size=8,
    #     alpha=0.8
    # )

# Add layer separators and labels
for layer in range(num_layers):
    x_pos = layer * layer_gap
    plt.axvline(x=x_pos, color='lightgray', linestyle='--', alpha=0.3)
    plt.text(x_pos, cls_base_y + 2, f'Layer {layer}', 
             ha='center', va='bottom', fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

# Add token type labels
plt.text(-layer_gap * 0.3, cls_base_y, 'CLS\nTokens', 
         ha='center', va='center', fontsize=14, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
plt.text(-layer_gap * 0.3, img_base_y, 'IMG\nTokens', 
         ha='center', va='center', fontsize=14, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# Create legend for concept levels
from matplotlib.lines import Line2D
legend_elements = []
for level, color in level_colors.items():
    # Find concept types for this level
    concept_types = []
    for files in concept_files[level]:
        concept_types.append(files.replace('.txt', ''))
    
    legend_elements.append(
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color, 
               markersize=12, label=f'Level {level}: {", ".join(concept_types)}')
    )

plt.legend(handles=legend_elements, loc='upper right', 
          bbox_to_anchor=(1.0, 1.0), fontsize=10)

# Customize the plot
plt.title(f"Original Concept Flow Graph (G_fixed)\n{G_fixed.number_of_nodes()} nodes, {G_fixed.number_of_edges()} edges", 
          fontsize=16, fontweight='bold', pad=20)
plt.axis('off')

# Set axis limits to show all content properly
plt.xlim(-layer_gap * 0.6, (num_layers) * layer_gap + layer_gap * 0.2)
plt.ylim(img_base_y - 3, cls_base_y + 4)

plt.tight_layout()
plt.show()

# Print some statistics about the graph
print(f"\nG_fixed Graph Statistics:")
print(f"  Total nodes: {G_fixed.number_of_nodes()}")
print(f"  Total edges: {G_fixed.number_of_edges()}")

# Show distribution by layers
nodes_by_layer = {}
for node in G_fixed.nodes():
    layer = node.split(":")[0]
    token_type = node.split(":")[1]
    nodes_by_layer.setdefault(layer, {}).setdefault(token_type, 0)
    nodes_by_layer[layer][token_type] += 1

print(f"\nNodes by layer:")
for layer in sorted(nodes_by_layer.keys(), key=lambda x: int(x[1:])):
    cls_count = nodes_by_layer[layer].get('CLS', 0)
    img_count = nodes_by_layer[layer].get('IMG', 0)
    print(f"  {layer}: {cls_count} CLS + {img_count} IMG = {cls_count + img_count} total")

# Show edge weight statistics
if G_fixed.number_of_edges() > 0:
    weights = [d['weight'] for u, v, d in G_fixed.edges(data=True)]
    print(f"\nEdge weight statistics:")
    print(f"  Min weight: {min(weights):.6f}")
    print(f"  Max weight: {max(weights):.6f}")
    print(f"  Mean weight: {np.mean(weights):.6f}")
    print(f"  Median weight: {np.median(weights):.6f}")

## Weighted by Prediction

In [ ]:
def measure_final_decision_impact_all_layers(
    image, text_queries, saes_cls, saes_img, hooked_model, freq_idx_cls, freq_idx_img, 
    layer_concepts_cls, layer_concepts_img, all_concepts=None, top_k=10
):
    """
    Measure end-to-end impact of concepts from all layers on final CLIP similarity scores.
    
    Args:
        all_concepts: Pre-computed concepts from get_top_k_concepts_all_layers (optional)
        Other args same as before
    
    Returns:
        all_cls_impacts: dict[layer] -> np.array[top_k, num_queries]
        all_img_impacts: dict[layer] -> np.array[top_k, num_queries] 
        all_cls_concepts: dict[layer] -> list[concept_names]
        all_img_concepts: dict[layer] -> list[concept_names]
        original_similarities: np.array[num_queries]
    """
    
    from transformers import CLIPModel, CLIPProcessor
    
    # Load full CLIP model for final similarity computation
    full_clip_model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32", 
        torch_dtype=hooked_model.torch_dtype
    ).to(hooked_model.device)
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    
    # Get original embeddings from all layers
    embeddings = hooked_model.get_activations_from_image(image)
    
    # Get original final vision embedding (baseline)
    original_vision_hidden = embeddings["layer-11_resid-post"][0, 0, :]
    original_vision_hidden_norm = full_clip_model.vision_model.post_layernorm(original_vision_hidden.unsqueeze(0)).squeeze(0)
    original_vision_embedding = full_clip_model.visual_projection(original_vision_hidden_norm)
    
    # Encode text queries
    text_inputs = clip_processor(text=text_queries, return_tensors="pt", padding=True).to(hooked_model.device)
    with torch.no_grad():
        text_embeddings = full_clip_model.get_text_features(**text_inputs)
        text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)
    
    original_vision_embedding_norm = original_vision_embedding / original_vision_embedding.norm(dim=-1, keepdim=True)
    original_similarities = torch.matmul(original_vision_embedding_norm, text_embeddings.T)
    original_similarities_np = original_similarities.float().detach().cpu().numpy()
    print(f"Original similarities: {original_similarities_np}")
    
    # Define baseline similarity (typical minimum for unrelated content)
    baseline_similarity = 0.2  # Typical similarity for unrelated image-text pairs
    print(f"Using baseline similarity: {baseline_similarity:.3f}")
    
    # Storage for results
    all_cls_impacts = {}
    all_img_impacts = {}
    all_cls_concepts = {}
    all_img_concepts = {}
    
    eps = 1e-8
    
    # Process each layer
    for layer_L in range(12):
        print(f"Processing layer {layer_L}...")
        
        layer_name_L = f"layer-{layer_L}_resid-post"
        
        # === CLS CONCEPT ABLATION (all layers) ===
        cls_activation_L = embeddings[layer_name_L][0, 0, :]
        sae_cls_L = saes_cls[layer_L]
        
        # Use pre-computed concepts if available, otherwise fall back to original logic
        if all_concepts is not None:
            # Use pre-computed top-k CLS concepts and indices
            cls_data = all_concepts[layer_L]['CLS']
            top_indices_cls = cls_data['indices']
            cls_concepts = cls_data['concepts']
            # Get raw feature activations for modification
            feature_acts_cls = sae_cls_L.encode(cls_activation_L.unsqueeze(0)).squeeze(0)
        else:
            # Original logic as fallback
            feature_acts_cls = sae_cls_L.encode(cls_activation_L.unsqueeze(0)).squeeze(0)
            feature_acts_cls_freq = feature_acts_cls.clone()
            feature_acts_cls_freq[freq_idx_cls[layer_L]] = 0
            top_values_cls, top_indices_tensor = torch.topk(feature_acts_cls_freq, top_k)
            nonzero_mask_cls = top_values_cls != 0
            top_indices_cls = top_indices_tensor[nonzero_mask_cls].cpu().numpy()
            
            # Get concept names for valid indices
            cls_concepts = []
            for idx in top_indices_cls:
                if idx < len(layer_concepts_cls[layer_L]):
                    cls_concepts.append(layer_concepts_cls[layer_L][idx])
                else:
                    cls_concepts.append(f"feat_{idx}")
        
        cls_impact = np.zeros((len(top_indices_cls), len(text_queries)))
        
        for i, idx in enumerate(top_indices_cls):
            # Ablate feature in current layer
            feature_acts_cls_ablated = feature_acts_cls.clone()
            feature_acts_cls_ablated[idx] = 0
            
            # Reconstruct CLS token at current layer
            ablated_cls_embedding_L = sae_cls_L.decode(feature_acts_cls_ablated.unsqueeze(0)).squeeze(0)
            
            # Create modified layer output
            modified_layer_L = embeddings[layer_name_L].clone()
            modified_layer_L[0, 0, :] = ablated_cls_embedding_L
            
            # Propagate forward through remaining layers
            current_output = modified_layer_L
            for next_layer in range(layer_L + 1, 12):
                current_output = hooked_model.model.encoder.layers[next_layer](
                    current_output, attention_mask=None, causal_attention_mask=None
                )
                if isinstance(current_output, tuple):
                    current_output = current_output[0]
            
            # Get final CLS token and compute similarity
            final_vision_hidden = current_output[0, 0, :]
            final_vision_hidden = final_vision_hidden.to(dtype=full_clip_model.dtype)
            
            final_vision_hidden_norm = full_clip_model.vision_model.post_layernorm(final_vision_hidden.unsqueeze(0)).squeeze(0)
            final_vision_embedding = full_clip_model.visual_projection(final_vision_hidden_norm)
            final_vision_embedding_norm = final_vision_embedding / final_vision_embedding.norm(dim=-1, keepdim=True)
            
            new_similarities = torch.matmul(final_vision_embedding_norm, text_embeddings.T)
            impact_absolute = (original_similarities - new_similarities).float().detach().cpu().numpy()
            
            # Apply ReLU to remove negative impacts
            impact_absolute = np.maximum(impact_absolute, 0)
            
            # Normalize by the range from baseline to original similarity
            similarity_range = original_similarities_np - baseline_similarity
            impact_normalized = np.where(similarity_range > eps, 
                                       impact_absolute / (similarity_range + eps), 
                                       0)
            cls_impact[i, :] = impact_normalized
        
        # === IMG CONCEPT ABLATION (layers 0-10 only) ===
        if layer_L < 11:
            # Use pre-computed concepts if available, otherwise fall back to original logic
            if all_concepts is not None:
                # Use pre-computed top-k IMG concepts and indices
                img_data = all_concepts[layer_L]['IMG']
                top_indices_img = img_data['indices']
                img_concepts = img_data['concepts']
                # Get raw feature activations for modification
                img_activations_L = embeddings[layer_name_L][0, 1:, :]
                sae_img_L = saes_img[layer_L]
                with torch.no_grad():
                    feature_acts_img = sae_img_L.encode(img_activations_L)
            else:
                # Original logic as fallback
                img_activations_L = embeddings[layer_name_L][0, 1:, :]
                sae_img_L = saes_img[layer_L]
                
                with torch.no_grad():
                    feature_acts_img = sae_img_L.encode(img_activations_L)
                    feature_acts_img_freq = feature_acts_img.clone()
                    feature_acts_img_freq[:, freq_idx_img[layer_L]] = 0
                    avg_feature_acts_img = feature_acts_img_freq.mean(dim=0)
                    top_values_img, top_indices_tensor = torch.topk(avg_feature_acts_img, top_k)
                    nonzero_mask_img = top_values_img != 0
                    top_indices_img = top_indices_tensor[nonzero_mask_img].cpu().numpy()
                
                # Get concept names for valid indices
                img_concepts = []
                for idx in top_indices_img:
                    if idx < len(layer_concepts_img[layer_L]):
                        img_concepts.append(layer_concepts_img[layer_L][idx])
                    else:
                        img_concepts.append(f"feat_{idx}")
            
            img_impact = np.zeros((len(top_indices_img), len(text_queries)))
            
            for i, idx in enumerate(top_indices_img):
                # Ablate feature in current layer
                feature_acts_img_ablated = feature_acts_img.clone()
                feature_acts_img_ablated[:, idx] = 0
                
                # Reconstruct image tokens at current layer
                ablated_img_embeddings_L = sae_img_L.decode(feature_acts_img_ablated)
                
                # Create modified layer output
                modified_layer_L = embeddings[layer_name_L].clone()
                modified_layer_L[0, 1:, :] = ablated_img_embeddings_L
                
                # Propagate forward through remaining layers
                current_output = modified_layer_L
                for next_layer in range(layer_L + 1, 12):
                    current_output = hooked_model.model.encoder.layers[next_layer](
                        current_output, attention_mask=None, causal_attention_mask=None
                    )
                    if isinstance(current_output, tuple):
                        current_output = current_output[0]
                
                # Get final CLS token and compute similarity
                final_vision_hidden = current_output[0, 0, :]
                final_vision_hidden = final_vision_hidden.to(dtype=full_clip_model.dtype)
                
                final_vision_hidden_norm = full_clip_model.vision_model.post_layernorm(final_vision_hidden.unsqueeze(0)).squeeze(0)
                final_vision_embedding = full_clip_model.visual_projection(final_vision_hidden_norm)
                final_vision_embedding_norm = final_vision_embedding / final_vision_embedding.norm(dim=-1, keepdim=True)
                
                new_similarities = torch.matmul(final_vision_embedding_norm, text_embeddings.T)
                impact_absolute = (original_similarities - new_similarities).float().detach().cpu().numpy()
                
                # Apply ReLU to remove negative impacts
                impact_absolute = np.maximum(impact_absolute, 0)
                
                # Normalize by the range from baseline to original similarity
                similarity_range = original_similarities_np - baseline_similarity
                impact_normalized = np.where(similarity_range > eps, 
                                           impact_absolute / (similarity_range + eps), 
                                           0)
                img_impact[i, :] = impact_normalized
            
            # Store IMG concept names
            all_img_concepts[layer_L] = img_concepts
        else:
            # Layer 11 IMG tokens don't affect final decision (no forward propagation)
            img_impact = np.zeros((0, len(text_queries)))  # Empty array
            all_img_concepts[layer_L] = []
            print(f"  Skipped IMG tokens (no forward propagation from layer 11)")
        
        # Store results for this layer
        all_cls_impacts[layer_L] = cls_impact
        all_img_impacts[layer_L] = img_impact
        
        # Store CLS concept names
        all_cls_concepts[layer_L] = cls_concepts
    
    return all_cls_impacts, all_img_impacts, all_cls_concepts, all_img_concepts, original_similarities_np

In [ ]:
# Define your text queries for evaluation
text_queries = [
    "dog"
]

all_cls_impacts, all_img_impacts, all_cls_concepts, all_img_concepts, original_sims = measure_final_decision_impact_all_layers(
    image, text_queries, saes_cls, saes_img, hooked_model, 
    freq_idx_cls, freq_idx_img, layer_concepts_cls, layer_concepts_img, 
    all_concepts=all_concepts,  # Use pre-computed concepts
    top_k=5
)

# Create concept impact map for visualization
concept_impact_map = {}

# Map each concept to its maximum impact across all queries
for layer in range(12):
    # CLS concepts
    for i, concept in enumerate(all_cls_concepts[layer]):
        if i < all_cls_impacts[layer].shape[0]:
            max_impact = all_cls_impacts[layer][i, :].max()
            node_name = f"L{layer}:CLS:{concept}"
            concept_impact_map[node_name] = max_impact
    
    # IMG concepts
    for i, concept in enumerate(all_img_concepts[layer]):
        if i < all_img_impacts[layer].shape[0]:
            max_impact = all_img_impacts[layer][i, :].max()
            node_name = f"L{layer}:IMG:{concept}"
            concept_impact_map[node_name] = max_impact

print(f"Created concept impact map with {len(concept_impact_map)} entries")

# Create re-weighted graph by multiplying original edge weights with target concept impacts
G_reweighted = nx.DiGraph()

# Add all nodes first
# G_reweighted.add_nodes_from(G.nodes())
G_reweighted.add_nodes_from(G_fixed.nodes())

# Re-weight edges based on target concept impact
# for u, v, data in G.edges(data=True):
for u, v, data in G_fixed.edges(data=True):
    original_weight = data.get('weight', 0)
    target_impact = concept_impact_map.get(v, 0)  # Impact of target concept
    
    # New weight = original influence × target concept's final decision impact
    new_weight = original_weight * target_impact
    
    # Only add edges with meaningful weights
    if new_weight > 1e-6:
        G_reweighted.add_edge(u, v, 
                             weight=new_weight, 
                             original_weight=original_weight, 
                             target_impact=target_impact)

print(f"Re-weighted graph: {G_reweighted.number_of_nodes()} nodes, {G_reweighted.number_of_edges()} edges")

# Add final decision node and connect from layer 11
query_name = text_queries[0].replace("a photo of ", "").replace("an image of ", "").strip()
final_decision_node = f"Final:DECISION:{query_name}"
G_reweighted.add_node(final_decision_node)

# Add edges from layer 11 concepts to final decision using end-to-end impact scores
for layer in [11]:  # Only layer 11 connects to final decision
    # CLS concepts from layer 11
    for i, concept in enumerate(all_cls_concepts[layer]):
        source_node = f"L{layer}:CLS:{concept}"
        if source_node in G_reweighted.nodes() and i < all_cls_impacts[layer].shape[0]:
            impact_score = all_cls_impacts[layer][i, 0]  # Use first query (index 0)
            if impact_score > 1e-6:  # Only add meaningful connections
                G_reweighted.add_edge(source_node, final_decision_node, 
                                    weight=impact_score, 
                                    original_weight=1.0,  # Direct connection
                                    target_impact=impact_score)
    
    # IMG concepts from layer 11 (if any - they should have zero impact)
    for i, concept in enumerate(all_img_concepts[layer]):
        source_node = f"L{layer}:IMG:{concept}"
        if source_node in G_reweighted.nodes() and i < all_img_impacts[layer].shape[0]:
            impact_score = all_img_impacts[layer][i, 0]  # Use first query (index 0)
            if impact_score > 1e-6:
                G_reweighted.add_edge(source_node, final_decision_node, 
                                    weight=impact_score, 
                                    original_weight=1.0,
                                    target_impact=impact_score)

print(f"Final re-weighted graph: {G_reweighted.number_of_nodes()} nodes, {G_reweighted.number_of_edges()} edges")

# Debug the weights to choose appropriate thresholds
if G_reweighted.number_of_edges() > 0:
    all_weights = [d['weight'] for u, v, d in G_reweighted.edges(data=True)]
    print(f"\nRe-weighted edge weights:")
    print(f"  Min: {min(all_weights):.8f}")
    print(f"  Max: {max(all_weights):.8f}")
    print(f"  Mean: {np.mean(all_weights):.8f}")
    print(f"  Median: {np.median(all_weights):.8f}")
    
    # Count how many edges survive different thresholds
    for thresh in [0.01, 0.001, 0.0001, 0.00001, 0.000001]:
        count = sum(1 for w in all_weights if w >= thresh)
        print(f"  Edges >= {thresh}: {count}")

In [ ]:
def plot_node_contributions_only_with_impact(embeddings, saes_cls, saes_img, freq_idx_cls, freq_idx_img, layer_concepts_cls, layer_concepts_img, concept_impact_map, all_concepts=None, top_k=5):
    """
    Plot only nodes using the SAME concept selection logic as your main code.
    Node size represents impact on final decision, numbers show impact values.
    
    Args:
        all_concepts: Pre-computed concepts from get_top_k_concepts_all_layers (optional)
        Other args same as before
    """
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Extract top activated features using the SAME logic as your main code
    layer_contributions = {}
    
    print("Extracting top activated features for plotting...")
    
    for layer in range(12):
        layer_name = f"layer-{layer}_resid-post"
        layer_activations = embeddings[layer_name]

        layer_contributions[layer] = {'CLS': [], 'IMG': []}

        # === CLS token processing ===
        if all_concepts is not None:
            # Use pre-computed CLS concepts and their indices
            cls_data = all_concepts[layer]['CLS']
            concepts_cls_L = cls_data['concepts'][:top_k]  # Slice to get only first top_k
            
            for concept in concepts_cls_L:
                node_name = f"L{layer}:CLS:{concept}"
                # Use impact from concept_impact_map
                impact_val = concept_impact_map.get(node_name, 0.0)
                
                layer_contributions[layer]['CLS'].append({
                    'layer': layer,
                    'type': 'CLS',
                    'concept': concept,
                    'impact': impact_val
                })
        else:
            # Original logic as fallback
            cls_activation = layer_activations[0, 0, :]
            sae_cls = saes_cls[layer]
            with torch.no_grad():
                feature_acts_cls = sae_cls.encode(cls_activation.unsqueeze(0)).squeeze(0)
                # Zero out frequent features
                feature_acts_cls[freq_idx_cls[layer]] = 0
                top_values_cls, top_indices_cls = torch.topk(feature_acts_cls, top_k)
                # Filter out zero activations
                nonzero_mask_cls = top_values_cls != 0
                
                # Get concepts for non-zero top features
                concepts_cls = layer_concepts_cls.get(layer, [])
                valid_indices_cls = top_indices_cls[nonzero_mask_cls].cpu().numpy()
                
                for feature_idx in valid_indices_cls:
                    if feature_idx < len(concepts_cls):
                        concept = concepts_cls[feature_idx]
                        node_name = f"L{layer}:CLS:{concept}"
                        # Use impact from concept_impact_map instead of activation
                        impact_val = concept_impact_map.get(node_name, 0.0)
                        
                        layer_contributions[layer]['CLS'].append({
                            'layer': layer,
                            'type': 'CLS',
                            'concept': concept,
                            'impact': impact_val
                        })

        # === IMG token processing ===
        if all_concepts is not None:
            # Use pre-computed IMG concepts and their indices
            img_data = all_concepts[layer]['IMG']
            concepts_img_L = img_data['concepts'][:top_k]  # Slice to get only first top_k
            
            for concept in concepts_img_L:
                node_name = f"L{layer}:IMG:{concept}"
                # Use impact from concept_impact_map
                impact_val = concept_impact_map.get(node_name, 0.0)
                
                layer_contributions[layer]['IMG'].append({
                    'layer': layer,
                    'type': 'IMG',
                    'concept': concept,
                    'impact': impact_val
                })
        else:
            # Original logic as fallback
            img_activations = layer_activations[0, 1:, :]
            avg_img_activation = img_activations.mean(dim=0)
            sae_img = saes_img[layer]
            with torch.no_grad():
                feature_acts_img = sae_img.encode(avg_img_activation.unsqueeze(0)).squeeze(0)
                # Zero out frequent features
                feature_acts_img[freq_idx_img[layer]] = 0
                top_values_img, top_indices_img = torch.topk(feature_acts_img, top_k)
                # Filter out zero activations
                nonzero_mask_img = top_values_img != 0
                
                # Get concepts for non-zero top features
                concepts_img = layer_concepts_img.get(layer, [])
                valid_indices_img = top_indices_img[nonzero_mask_img].cpu().numpy()
                
                for feature_idx in valid_indices_img:
                    if feature_idx < len(concepts_img):
                        concept = concepts_img[feature_idx]
                        node_name = f"L{layer}:IMG:{concept}"
                        # Use impact from concept_impact_map instead of activation
                        impact_val = concept_impact_map.get(node_name, 0.0)
                        
                        layer_contributions[layer]['IMG'].append({
                            'layer': layer,
                            'type': 'IMG',
                            'concept': concept,
                            'impact': impact_val
                        })
    
    # Filter out zero-impact contributions for cleaner visualization
    filtered_contributions = {}
    all_contributions = []
    
    for layer in sorted(layer_contributions.keys()):
        filtered_contributions[layer] = {'CLS': [], 'IMG': []}
        
        # Filter CLS contributions with non-zero impact
        for contrib in layer_contributions[layer]['CLS']:
            if contrib['impact'] > 0:  # Only show concepts with positive impact
                filtered_contributions[layer]['CLS'].append(contrib)
        
        # Filter IMG contributions with non-zero impact
        for contrib in layer_contributions[layer]['IMG']:
            if contrib['impact'] > 0:  # Only show concepts with positive impact
                filtered_contributions[layer]['IMG'].append(contrib)
    
    # Create positions for remaining contributions
    cls_base_y = 3.0
    img_base_y = 1.0
    vertical_spacing = 0.6
    
    for layer in sorted(filtered_contributions.keys()):
        # Handle CLS nodes for this layer
        cls_nodes = filtered_contributions[layer]['CLS']
        for i, contrib in enumerate(cls_nodes):
            y_pos = cls_base_y + (i - len(cls_nodes)/2 + 0.5) * vertical_spacing
            contrib['y_pos'] = y_pos
            all_contributions.append(contrib)
        
        # Handle IMG nodes for this layer
        img_nodes = filtered_contributions[layer]['IMG']
        for i, contrib in enumerate(img_nodes):
            y_pos = img_base_y + (i - len(img_nodes)/2 + 0.5) * vertical_spacing
            contrib['y_pos'] = y_pos
            all_contributions.append(contrib)
    
    if not all_contributions:
        print("No contributions with positive impact found")
        return
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(20, 12))
    
    # Calculate node sizes based on impact values
    max_impact = max(c['impact'] for c in all_contributions)
    min_impact = min(c['impact'] for c in all_contributions)
    
    # Plot each contribution with size proportional to impact
    for contrib in all_contributions:
        x = contrib['layer']
        y = contrib['y_pos']
        concept = contrib['concept']
        token_type = contrib['type']
        impact = contrib['impact']
        
        # Node size proportional to impact value
        if max_impact > min_impact:
            size_ratio = (impact - min_impact) / (max_impact - min_impact)
        else:
            size_ratio = 0.5
        node_size = 200 + size_ratio * 800  # Scale from 200 to 1000
        
        # Color based on token type
        color = '#1f77b4' if token_type == 'CLS' else '#2ca02c'
        
        # Plot the node
        ax.scatter(x, y, s=node_size, c=color, alpha=0.7, edgecolors='black', linewidth=1)
        
        # Show concept name and impact value
        ax.annotate(f"{concept}\n{impact:.4f}", 
                   (x, y), 
                   xytext=(0, 0), 
                   textcoords='offset points',
                   ha='center', 
                   va='center',
                   fontsize=8,
                   fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))
    
    # Customize plot
    ax.set_xlabel('Layer', fontsize=14, fontweight='bold')
    ax.set_ylabel('Token Type Region', fontsize=14, fontweight='bold')
    ax.set_title(f'Top-{top_k} Activated Concepts (Node Size ∝ Final Decision Impact)', 
                fontsize=16, fontweight='bold')
    
    ax.set_ylim(-1, 5.5)
    ax.axhline(y=2.25, color='gray', linestyle='--', alpha=0.5)
    ax.text(-0.5, 3.0, 'CLS\nTokens', fontsize=12, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))
    ax.text(-0.5, 1.0, 'IMG\nTokens', fontsize=12, fontweight='bold', 
            ha='center', va='center', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
    
    ax.set_xlim(-0.8, 11.8)
    ax.set_xticks(range(12))
    ax.set_xticklabels([f'L{i}' for i in range(12)])
    
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', 
               markersize=10, label='CLS tokens'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ca02c', 
               markersize=10, label='IMG tokens'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
               markersize=5, label='Lower impact'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', 
               markersize=15, label='Higher impact')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    # Add impact statistics
    ax.text(0.02, 0.98, 
           f"Max impact: {max_impact:.4f}\nMin impact: {min_impact:.4f}\nMean impact: {np.mean([c['impact'] for c in all_contributions]):.4f}", 
           transform=ax.transAxes, 
           verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7),
           fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Print the concepts for verification WITH impact values
    print(f"\nShowing Top-{top_k} Activated Concepts (with impact values):")
    for layer in sorted(filtered_contributions.keys()):
        if filtered_contributions[layer]['CLS']:
            cls_concepts = [(c['concept'], c['impact']) for c in filtered_contributions[layer]['CLS']]
            print(f"L{layer} CLS: {cls_concepts}")
        if filtered_contributions[layer]['IMG']:
            img_concepts = [(c['concept'], c['impact']) for c in filtered_contributions[layer]['IMG']]
            print(f"L{layer} IMG: {img_concepts}")
    
    # Print top contributors ranked by impact
    sorted_contribs = sorted(all_contributions, key=lambda x: x['impact'], reverse=True)
    print(f"\nTop 10 Contributors by Final Decision Impact:")
    print(f"{'Rank':<4} {'Layer':<5} {'Type':<4} {'Impact':<12} {'Concept'}")
    print("-" * 60)
    for rank, contrib in enumerate(sorted_contribs[:10], 1):
        print(f"{rank:<4} {contrib['layer']:<5} {contrib['type']:<4} "
              f"{contrib['impact']:<12.6f} {contrib['concept']}")

In [ ]:
# Use the impact-based version with pre-computed concepts
plot_node_contributions_only_with_impact(
    embeddings, saes_cls, saes_img, freq_idx_cls, freq_idx_img, 
    layer_concepts_cls, layer_concepts_img, concept_impact_map, 
    all_concepts=all_concepts,  # Use pre-computed concepts
    top_k=top_k_global
)

In [ ]:
# FIXED: Add visualization for the re-weighted graph with final decision node
import random
import matplotlib.pyplot as plt

# Set random seed for consistent layout
random.seed(42)

# FIXED: Use node_pos_fixed instead of creating new positions
# The node_pos_fixed was created when building G_fixed and matches its nodes
node_pos_reweighted = node_pos_fixed.copy()  # Use the positions from G_fixed

layer_gap = 4
cls_gap = 2.5
img_gap = 2.5
cls_base_y = top_k * 3
img_base_y = 0
num_layers = 12

# Add final decision node position
query_name = text_queries[0].replace("a photo of ", "").replace("an image of ", "").strip()
final_decision_node = f"Final:DECISION:{query_name}"
if final_decision_node in G_reweighted.nodes():
    node_pos_reweighted[final_decision_node] = (num_layers * layer_gap, (cls_base_y + img_base_y) / 2)

# Apply edge weight threshold for visualization
weight_threshold_viz = 0.00  # Adjust this threshold as needed
edges_to_remove_viz = [(u, v) for u, v, d in G_reweighted.edges(data=True) 
                       if d.get("weight", 0) < weight_threshold_viz]
G_viz = G_reweighted.copy()
G_viz.remove_edges_from(edges_to_remove_viz)

print(f"Visualization graph: {G_viz.number_of_nodes()} nodes, {G_viz.number_of_edges()} edges")
print(f"Final decision node: {query_name}")

# FIXED: Only include nodes that actually exist in G_viz AND have positions
nodes_with_positions = [node for node in G_viz.nodes() if node in node_pos_reweighted]
missing_nodes = [node for node in G_viz.nodes() if node not in node_pos_reweighted]

if missing_nodes:
    print(f"Warning: {len(missing_nodes)} nodes missing positions:")
    for node in missing_nodes[:5]:  # Show first 5
        print(f"  {node}")
    if len(missing_nodes) > 5:
        print(f"  ... and {len(missing_nodes) - 5} more")

# Filter G_viz to only include nodes with positions
G_viz_filtered = G_viz.subgraph(nodes_with_positions).copy()
print(f"Filtered visualization graph: {G_viz_filtered.number_of_nodes()} nodes, {G_viz_filtered.number_of_edges()} edges")

# Create node colors based on concept levels (same as before, plus special color for final decision)
node_colors_reweighted = []
for node in G_viz_filtered.nodes():
    if node.startswith("Final:"):
        node_colors_reweighted.append("#ff69b4")  # Pink for final decision
    else:
        concept = node.split(":")[-1]
        level = concept_level_map.get(concept, None)
        color = level_colors.get(level, "#cccccc")
        node_colors_reweighted.append(color)

# Visualize with matplotlib
plt.figure(figsize=(layer_gap * (num_layers + 1), 16))

# Draw nodes with different sizes (larger for final decision)
node_sizes = [1200 if node.startswith("Final:") else 800 for node in G_viz_filtered.nodes()]

nx.draw_networkx_nodes(
    G_viz_filtered,
    pos=node_pos_reweighted,
    node_color=node_colors_reweighted,
    node_size=node_sizes,
)

# CREATE CUSTOM LABELS SHOWING ONLY CONCEPT NAMES
node_labels = {}
for node in G_viz_filtered.nodes():
    if node.startswith("Final:"):
        # For final decision node, show only the query name
        node_labels[node] = node.split(":")[-1]  # e.g., "school bus"
    else:
        # For regular nodes, show only the concept name (last part after ":")
        node_labels[node] = node.split(":")[-1]  # e.g., "ball" instead of "L5:CLS:ball"

# Draw node labels with only concept names
nx.draw_networkx_labels(
    G_viz_filtered,
    pos=node_pos_reweighted,
    labels=node_labels,  # Use custom labels instead of default node names
    font_size=18,
    font_weight='bold',
    font_color='black'
)

# Draw edges with thickness proportional to re-weighted values
if G_viz_filtered.number_of_edges() > 0:
    edges = G_viz_filtered.edges(data=True)
    max_weight = max([d['weight'] for (_, _, d) in edges])
    edge_widths = [max(0.5, min(8, d['weight'] * 8 / max_weight)) for (_, _, d) in edges]
    
    nx.draw_networkx_edges(
        G_viz_filtered,
        pos=node_pos_reweighted,
        edgelist=list(edges),
        width=edge_widths,
        edge_color='gray',
        arrows=True,
        arrowsize=20,
        alpha=0.7
    )
    
    # Draw edge labels with re-weighted values
    edge_labels = {(u, v): f"{d['weight']:.3f}" for u, v, d in edges}
    nx.draw_networkx_edge_labels(
        G_viz_filtered,
        pos=node_pos_reweighted,
        edge_labels=edge_labels,
        font_size=16
    )

plt.title(f"Re-weighted Concept Flow to '{query_name}' Decision\n(Edge Weight × Target Impact, Threshold: {weight_threshold_viz})")
plt.axis('off')
plt.tight_layout()
plt.show()

## Remove Dead Nodes

In [ ]:
def remove_dead_end_nodes(G, final_decision_nodes=None):
    """
    Recursively remove nodes that have no outgoing edges (dead ends) from the graph.
    Preserves final decision nodes even if they have no outgoing edges.
    
    Args:
        G: NetworkX DiGraph
        final_decision_nodes: list of node names that should be preserved (e.g., final decision nodes)
    
    Returns:
        cleaned_G: NetworkX DiGraph with dead end nodes removed
        removed_nodes: list of removed node names for debugging
    """
    import networkx as nx
    
    # Create a copy to avoid modifying the original graph
    cleaned_G = G.copy()
    removed_nodes = []
    
    # Set of nodes to preserve (final decision nodes)
    preserve_nodes = set(final_decision_nodes) if final_decision_nodes else set()
    
    # Keep removing dead end nodes until no more can be removed
    nodes_removed_this_iteration = True
    iteration = 0
    
    while nodes_removed_this_iteration:
        nodes_removed_this_iteration = False
        iteration += 1
        
        # Find nodes with no outgoing edges
        dead_end_nodes = []
        for node in cleaned_G.nodes():
            # Skip preserved nodes (like final decision nodes)
            if node in preserve_nodes:
                continue
                
            # Check if node has no outgoing edges
            if cleaned_G.out_degree(node) == 0:
                dead_end_nodes.append(node)
        
        # Remove dead end nodes
        if dead_end_nodes:
            print(f"Iteration {iteration}: Removing {len(dead_end_nodes)} dead end nodes")
            for node in dead_end_nodes:
                print(f"  Removing: {node}")
                cleaned_G.remove_node(node)
                removed_nodes.append(node)
            nodes_removed_this_iteration = True
        else:
            print(f"Iteration {iteration}: No more dead end nodes found")
    
    print(f"\nCleaning summary:")
    print(f"  Original nodes: {G.number_of_nodes()}")
    print(f"  Cleaned nodes: {cleaned_G.number_of_nodes()}")
    print(f"  Removed nodes: {len(removed_nodes)}")
    print(f"  Original edges: {G.number_of_edges()}")
    print(f"  Cleaned edges: {cleaned_G.number_of_edges()}")
    
    return cleaned_G, removed_nodes

In [ ]:
def visualize_cleaned_graph(G_original, text_queries, weight_threshold_viz=0.00):
    """
    Create a cleaned visualization of the re-weighted graph by removing dead end nodes.
    """
    import matplotlib.pyplot as plt
    import networkx as nx
    import random
    
    # Set random seed for consistent layout
    random.seed(42)
    
    # Apply edge weight threshold first
    edges_to_keep = [(u, v, d) for u, v, d in G_original.edges(data=True) 
                     if d.get("weight", 0) >= weight_threshold_viz]
    
    G_filtered = nx.DiGraph()
    G_filtered.add_edges_from([(u, v, d) for u, v, d in edges_to_keep])
    
    # Identify final decision nodes
    query_name = text_queries[0].replace("a photo of ", "").replace("an image of ", "").strip()
    final_decision_nodes = [f"Final:DECISION:{query_name}"]
    
    # Clean the graph by removing dead end nodes
    G_cleaned, removed_nodes = remove_dead_end_nodes(G_filtered, final_decision_nodes)
    
    print(f"\nVisualization graph after cleaning:")
    print(f"  Nodes: {G_cleaned.number_of_nodes()}")
    print(f"  Edges: {G_cleaned.number_of_edges()}")
    print(f"  Final decision node: {query_name}")
    
    if G_cleaned.number_of_nodes() == 0:
        print("Warning: No nodes left after cleaning!")
        return G_cleaned, removed_nodes
    
    # Create node positions for remaining nodes
    node_pos_cleaned = {}
    layer_gap = 4
    cls_gap = 2.5
    img_gap = 2.5
    cls_base_y = 15  # Increased to accommodate more nodes
    img_base_y = 0
    num_layers = 12
    
    # Group nodes by layer for positioning
    layer_nodes = {}
    for node in G_cleaned.nodes():
        if node.startswith("Final:"):
            layer_nodes.setdefault("Final", []).append(node)
        else:
            layer = node.split(":")[0]
            layer_nodes.setdefault(layer, []).append(node)
    
    # Position nodes layer by layer
    for layer, nodes in layer_nodes.items():
        if layer == "Final":
            # Position final decision node
            for node in nodes:
                node_pos_cleaned[node] = (num_layers * layer_gap, (cls_base_y + img_base_y) / 2)
        else:
            layer_num = int(layer[1:])  # Extract layer number from "L0", "L1", etc.
            
            # Separate CLS and IMG nodes
            cls_nodes = [n for n in nodes if ":CLS:" in n]
            img_nodes = [n for n in nodes if ":IMG:" in n]
            
            # Position CLS nodes
            for i, node in enumerate(cls_nodes):
                jitter = random.uniform(-0.3, 0.3)
                node_pos_cleaned[node] = (layer_num * layer_gap, cls_base_y - i * cls_gap + jitter)
            
            # Position IMG nodes
            for i, node in enumerate(img_nodes):
                jitter = random.uniform(-0.3, 0.3)
                node_pos_cleaned[node] = (layer_num * layer_gap, img_base_y - i * img_gap + jitter)
    
    # Create node colors based on concept levels
    node_colors_cleaned = []
    for node in G_cleaned.nodes():
        if node.startswith("Final:"):
            node_colors_cleaned.append("#ff69b4")  # Pink for final decision
        else:
            concept = node.split(":")[-1]
            level = concept_level_map.get(concept, None)
            color = level_colors.get(level, "#cccccc")
            node_colors_cleaned.append(color)
    
    # Create the visualization
    plt.figure(figsize=(layer_gap * (num_layers + 1), 16))
    
    # Draw nodes with different sizes (larger for final decision)
    node_sizes = [1200 if node.startswith("Final:") else 800 for node in G_cleaned.nodes()]
    
    nx.draw_networkx_nodes(
        G_cleaned,
        pos=node_pos_cleaned,
        node_color=node_colors_cleaned,
        node_size=node_sizes,
    )
    
    # Create custom labels showing only concept names
    node_labels = {}
    for node in G_cleaned.nodes():
        if node.startswith("Final:"):
            node_labels[node] = node.split(":")[-1]  # e.g., "school bus"
        else:
            node_labels[node] = node.split(":")[-1]  # e.g., "yellow" instead of "L5:CLS:yellow"
    
    # Draw node labels
    nx.draw_networkx_labels(
        G_cleaned,
        pos=node_pos_cleaned,
        labels=node_labels,
        font_size=16,  # Slightly smaller since we have fewer nodes now
        font_weight='bold',
        font_color='black'
    )
    
    # Draw edges with thickness proportional to re-weighted values
    if G_cleaned.number_of_edges() > 0:
        edges = G_cleaned.edges(data=True)
        max_weight = max([d['weight'] for (_, _, d) in edges])
        edge_widths = [max(0.5, min(8, d['weight'] * 8 / max_weight)) for (_, _, d) in edges]
        
        nx.draw_networkx_edges(
            G_cleaned,
            pos=node_pos_cleaned,
            edgelist=list(edges),
            width=edge_widths,
            edge_color='gray',
            arrows=True,
            arrowsize=20,
            alpha=0.7
        )
        
        # Draw edge labels with re-weighted values
        edge_labels = {(u, v): f"{d['weight']:.3f}" for u, v, d in edges}
        nx.draw_networkx_edge_labels(
            G_cleaned,
            pos=node_pos_cleaned,
            edge_labels=edge_labels,
            font_size=14  # Slightly smaller font for edge labels
        )
    
    plt.title(f"Cleaned Concept Flow to '{query_name}' Decision\n(Dead End Nodes Removed, Threshold: {weight_threshold_viz})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return G_cleaned, removed_nodes

In [ ]:
# Apply the cleaning function to your re-weighted graph
print("=== CLEANING DEAD END NODES ===")
G_cleaned, removed_nodes = visualize_cleaned_graph(G_reweighted, text_queries, weight_threshold_viz=0.0)

# Show some statistics about removed nodes
if removed_nodes:
    print(f"\nRemoved {len(removed_nodes)} dead end nodes:")
    
    # Group removed nodes by layer for better understanding
    removed_by_layer = {}
    for node in removed_nodes:
        if node.startswith("Final:"):
            layer = "Final"
        else:
            layer = node.split(":")[0]
        removed_by_layer.setdefault(layer, []).append(node)
    
    for layer in sorted(removed_by_layer.keys(), key=lambda x: int(x[1:]) if x.startswith("L") else 999):
        nodes = removed_by_layer[layer]
        print(f"  {layer}: {len(nodes)} nodes")
        for node in nodes[:3]:  # Show first 3 examples
            print(f"    {node}")
        if len(nodes) > 3:
            print(f"    ... and {len(nodes) - 3} more")

In [ ]:
def create_cleaned_graphviz_with_activations(G_original, text_queries, embeddings, saes_cls, saes_img, freq_idx_cls, freq_idx_img, layer_concepts_cls, layer_concepts_img, all_concepts=None, weight_threshold=0.001):
    """
    Create a Graphviz visualization of the cleaned graph showing node activation values instead of decision contributions.
    
    Args:
        all_concepts: Pre-computed concepts from get_top_k_concepts_all_layers (optional)
        Other args same as before
    """
    import graphviz
    import networkx as nx
    import torch
    
    # Apply edge weight threshold first
    edges_to_keep = [(u, v, d) for u, v, d in G_original.edges(data=True) 
                     if d.get("weight", 0) >= weight_threshold]
    
    G_filtered = nx.DiGraph()
    G_filtered.add_edges_from([(u, v, d) for u, v, d in edges_to_keep])
    
    # Identify final decision nodes
    query_name = text_queries[0].replace("a photo of ", "").replace("an image of ", "").strip()
    final_decision_nodes = [f"Final:DECISION:{query_name}"]
    
    # Clean the graph by removing dead end nodes
    G_cleaned, removed_nodes = remove_dead_end_nodes(G_filtered, final_decision_nodes)
    
    print(f"Cleaned graph for Graphviz: {G_cleaned.number_of_nodes()} nodes, {G_cleaned.number_of_edges()} edges")
    
    if G_cleaned.number_of_edges() == 0:
        print("Warning: No edges left after cleaning!")
        return None

    activation_map = {}
    activation_indices_map = {}  # Store concept indices separately
    
    # Calculate activation values for each layer and token type
    for layer in range(12):
        layer_name = f"layer-{layer}_resid-post"
        
        # === CLS activations ===
        if all_concepts is not None:
            # Use pre-computed CLS concepts and their indices and values
            cls_data = all_concepts[layer]['CLS']
            concepts_cls = cls_data['concepts']
            indices_cls = cls_data['indices'] 
            values_cls = cls_data['values']
            
            # Map concept names to activation values AND indices
            for concept, idx, activation_val in zip(concepts_cls, indices_cls, values_cls):
                node_name = f"L{layer}:CLS:{concept}"
                activation_map[node_name] = activation_val
                activation_indices_map[node_name] = idx
        else:
            # Original logic as fallback
            cls_activation = embeddings[layer_name][0, 0, :]
            sae_cls = saes_cls[layer]
            with torch.no_grad():
                feature_acts_cls = sae_cls.encode(cls_activation.unsqueeze(0)).squeeze(0)
                feature_acts_cls[freq_idx_cls[layer]] = 0  # Zero out frequent features
                
                # Get top concepts for this layer
                top_values_cls, top_indices_cls = torch.topk(feature_acts_cls, k=20)  # Get more than we need
                nonzero_mask_cls = top_values_cls != 0
                top_indices_cls = top_indices_cls[nonzero_mask_cls].cpu().numpy()
                top_values_cls = top_values_cls[nonzero_mask_cls].cpu().numpy()
                
                # Map concept names to activation values AND indices
                for idx, activation_val in zip(top_indices_cls, top_values_cls):
                    if idx < len(layer_concepts_cls[layer]):
                        concept = layer_concepts_cls[layer][idx]
                    else:
                        concept = f"feat_{idx}"
                    node_name = f"L{layer}:CLS:{concept}"
                    activation_map[node_name] = activation_val
                    activation_indices_map[node_name] = idx
        
        # === IMG activations ===
        if all_concepts is not None:
            # Use pre-computed IMG concepts and their indices and values
            img_data = all_concepts[layer]['IMG']
            concepts_img = img_data['concepts']
            indices_img = img_data['indices']
            values_img = img_data['values']
            
            # Map concept names to activation values AND indices
            for concept, idx, activation_val in zip(concepts_img, indices_img, values_img):
                node_name = f"L{layer}:IMG:{concept}"
                activation_map[node_name] = activation_val
                activation_indices_map[node_name] = idx
        else:
            # Original logic as fallback
            img_activations = embeddings[layer_name][0, 1:, :]
            sae_img = saes_img[layer]
            with torch.no_grad():
                feature_acts_img = sae_img.encode(img_activations)  # [num_img_tokens, dict_size]
                feature_acts_img[:, freq_idx_img[layer]] = 0  # Zero out frequent features
                avg_feature_acts_img = feature_acts_img.mean(dim=0)  # [dict_size]
                
                # Get top concepts for this layer
                top_values_img, top_indices_img = torch.topk(avg_feature_acts_img, k=20)  # Get more than we need
                nonzero_mask_img = top_values_img != 0
                top_indices_img = top_indices_img[nonzero_mask_img].cpu().numpy()
                top_values_img = top_values_img[nonzero_mask_img].cpu().numpy()
                
                # Map concept names to activation values AND indices
                for idx, activation_val in zip(top_indices_img, top_values_img):
                    if idx < len(layer_concepts_img[layer]):
                        concept = layer_concepts_img[layer][idx]
                    else:
                        concept = f"feat_{idx}"
                    node_name = f"L{layer}:IMG:{concept}"
                    activation_map[node_name] = activation_val
                    activation_indices_map[node_name] = idx
    
    # Create Graphviz diagram
    dot = graphviz.Digraph(engine="dot")
    dot.attr(
        rankdir="LR", 
        nodesep="0.8",  # Increased spacing since we have fewer nodes
        ranksep="2.0",  # Increased spacing between layers
        dpi="100",
        margin="0.3",
        concentrate="false",
        splines="spline",
        overlap="false"
    )
    
    # Organize remaining nodes by layer and type
    layer_nodes = {}
    final_nodes = set()
    
    for node in G_cleaned.nodes():
        if node.startswith("Final:"):
            final_nodes.add(node)
        else:
            layer = node.split(":")[0]
            typ = node.split(":")[1]
            layer_nodes.setdefault(layer, {}).setdefault(typ, []).append(node)
    
    node_id_map = {}
    
    # Create subgraphs for each layer that has remaining nodes
    for layer in sorted(layer_nodes.keys(), key=lambda x: int(x[1:])):
        type_dict = layer_nodes[layer]
        
        with dot.subgraph(name=f"cluster_{layer}") as layer_sg:
            layer_sg.attr(label=f"Layer {layer[1:]}", style="dashed", fontsize="18", fontweight="bold")
            
            # CLS subcluster
            if "CLS" in type_dict and type_dict["CLS"]:
                with layer_sg.subgraph(name=f"cluster_{layer}_CLS") as cls_sg:
                    cls_sg.attr(label="CLS", style="solid", color="#1f77b4", fontsize="18")
                    for node in type_dict["CLS"]:
                        idx = len(node_id_map)
                        node_id = f"n{idx}"
                        node_id_map[node] = node_id
                        concept = node.split(":")[-1]
                        
                        # Get activation value for display
                        activation_val = activation_map.get(node, 0.0)
                        
                        # Color based on concept level
                        level = concept_level_map.get(concept, None)
                        base_color = level_colors.get(level, "#cccccc")
                        
                        # Show concept name and activation value
                        display_concept = concept
                        
                        cls_sg.node(
                            node_id,
                            label=f"Idx: {activation_indices_map.get(node, 'N/A')}\\nAct: {activation_val:.3f}\\n\\n{display_concept}",
                            shape="box",
                            style="rounded,filled",
                            fillcolor=base_color,
                            fontname="Arial",
                            fontsize="20",
                            width="1.8",
                            height="1.4",
                            margin="0.15",
                            penwidth="1.5"
                        )
            
            # IMG subcluster
            if "IMG" in type_dict and type_dict["IMG"]:
                with layer_sg.subgraph(name=f"cluster_{layer}_IMG") as img_sg:
                    img_sg.attr(label="IMG", style="solid", color="#2ca02c", fontsize="12")
                    for node in type_dict["IMG"]:
                        idx = len(node_id_map)
                        node_id = f"n{idx}"
                        node_id_map[node] = node_id
                        concept = node.split(":")[-1]
                        
                        # Get activation value for display
                        activation_val = activation_map.get(node, 0.0)
                        
                        # Color based on concept level
                        level = concept_level_map.get(concept, None)
                        base_color = level_colors.get(level, "#cccccc")
                        
                        # Show concept name and activation value
                        display_concept = concept
                        
                        img_sg.node(
                            node_id,
                            label=f"Idx: {activation_indices_map.get(node, 'N/A')}\\nAct: {activation_val:.3f}\\n\\n{display_concept}",
                            shape="box",
                            style="rounded,filled",
                            fillcolor=base_color,
                            fontname="Arial",
                            fontsize="20",
                            width="1.8",
                            height="1.4",
                            margin="0.15",
                            penwidth="1.5"
                        )
    
    # Create final decision cluster
    if final_nodes:
        with dot.subgraph(name="cluster_final") as final_sg:
            final_sg.attr(label="Final Decision", style="dashed", fontsize="14", 
                         fontweight="bold", color="#ff69b4")
            for node in final_nodes:
                idx = len(node_id_map)
                node_id = f"n{idx}"
                node_id_map[node] = node_id
                concept = node.split(":")[-1]
                
                final_sg.node(
                    node_id,
                    label=concept,
                    shape="ellipse",
                    style="filled",
                    fillcolor="#ff69b4",
                    fontname="Arial",
                    fontsize="22",
                    fontweight="bold",
                    width="2.5",
                    height="1.5",
                    margin="0.2",
                    penwidth="3"
                )
    
    # # Add edges with weights proportional to re-weighted values
    # edges_data = list(G_cleaned.edges(data=True))
    # if edges_data:
    #     max_weight = max([d['weight'] for u, v, d in edges_data])
        
    #     for u, v, d in edges_data:
    #         weight = d['weight']
            
    #         # Edge thickness proportional to weight
    #         thickness = max(1.0, min(5.0, weight * 5 / max_weight))
            
    #         # Different edge color for final decision edges
    #         edge_color = "#ff1493" if v.startswith("Final:") else "#7C7C7C"
            
    #         # Font size based on weight importance
    #         font_size = "10" if weight > max_weight * 0.5 else "8"
            
    #         dot.edge(
    #             node_id_map[u], 
    #             node_id_map[v], 
    #             label=f"{weight:.3f}",
    #             color=edge_color,
    #             penwidth=str(thickness),
    #             fontsize=font_size,
    #             fontcolor=edge_color,
    #             arrowsize="0.8",
    #             arrowhead="normal"
    #         )
    
    # Add edges with weights proportional to re-weighted values
    edges_data = list(G_cleaned.edges(data=True))
    if edges_data:
        # Separate edges by type
        inter_layer_edges = [(u, v, d) for u, v, d in edges_data if not v.startswith("Final:")]
        final_edges = [(u, v, d) for u, v, d in edges_data if v.startswith("Final:")]
        
        # Calculate max weights separately
        max_inter_weight = max([d['weight'] for u, v, d in inter_layer_edges]) if inter_layer_edges else 1.0
        max_final_weight = max([d['weight'] for u, v, d in final_edges]) if final_edges else 1.0
        
        # Draw inter-layer edges with their own scaling
        for u, v, d in inter_layer_edges:
            weight = d['weight']
            thickness = max(1.0, min(5.0, weight * 5 / max_inter_weight))
            
            dot.edge(
                node_id_map[u], 
                node_id_map[v], 
                label=f"{weight:.3f}",
                color="#7C7C7C",
                penwidth=str(thickness),
                fontsize="8",
                fontcolor="#7C7C7C",
                arrowsize="0.8",
                arrowhead="normal"
            )
        
        # Draw final decision edges with their own scaling
        for u, v, d in final_edges:
            weight = d['weight']
            # thickness = max(0.3, min(5.0, weight * 20 / max_final_weight))
            weight_normalized = weight / max_final_weight  # 0.87 vs 1.0
            thickness = max(1.0, np.power(weight_normalized, 3) * 8)
            
            dot.edge(
                node_id_map[u], 
                node_id_map[v], 
                label=f"{weight:.3f}",
                color="#ff1493",
                penwidth=str(thickness),
                fontsize="10",
                fontcolor="#ff1493",
                arrowsize="0.8",
                arrowhead="normal"
            )
    
    # Add title
    dot.attr(label=f"Cleaned Concept Flow to '{query_name}' (Showing Activation Values)", 
             fontsize="18", fontweight="bold", labelloc="top", labeljust="center")
    
    return dot, G_cleaned, removed_nodes, activation_map, activation_indices_map

In [ ]:
def auto_determine_weight_threshold(G, method="percentile", **kwargs):
    """
    Automatically determine weight threshold based on edge weight distribution.
    
    Args:
        G: NetworkX graph with edge weights
        method: str, one of ["percentile", "std", "median_mad", "knee", "top_k", "adaptive"]
        **kwargs: Additional parameters for specific methods
    
    Returns:
        float: Automatically determined threshold
    """
    import numpy as np
    
    if G.number_of_edges() == 0:
        return 0.0
    
    # Extract all edge weights
    weights = np.array([d['weight'] for u, v, d in G.edges(data=True)])
    
    if method == "percentile":
        # Keep top X percentile of edges (default: top 20%)
        percentile = kwargs.get('percentile', 80)  # 80th percentile = top 20%
        threshold = np.percentile(weights, percentile)
        print(f"Percentile method ({percentile}th percentile): threshold = {threshold:.6f}")
    
    elif method == "std":
        # Keep edges above mean + k*std (default: mean + 1*std)
        k = kwargs.get('k', 1.0)
        threshold = np.mean(weights) + k * np.std(weights)
        print(f"Standard deviation method (mean + {k}*std): threshold = {threshold:.6f}")
    
    elif method == "median_mad":
        # Keep edges above median + k*MAD (Median Absolute Deviation)
        k = kwargs.get('k', 2.0)
        median = np.median(weights)
        mad = np.median(np.abs(weights - median))
        threshold = median + k * mad
        print(f"Median + MAD method (median + {k}*MAD): threshold = {threshold:.6f}")
    
    elif method == "top_k":
        # Keep only the top K strongest edges
        k = kwargs.get('k', 80)
        sorted_weights = np.sort(weights)[::-1]  # Sort descending
        if len(sorted_weights) >= k:
            threshold = sorted_weights[k-1]  # k-th strongest edge
        else:
            threshold = sorted_weights[-1] if len(sorted_weights) > 0 else 0.0
        print(f"Top-K method (top {k} edges): threshold = {threshold:.6f}")
    
    elif method == "knee":
        # Find "knee" point in sorted weights
        sorted_weights = np.sort(weights)[::-1]  # Sort descending
        
        if len(sorted_weights) > 10:
            # Simple knee detection: find point where second derivative changes most
            diffs = np.diff(sorted_weights)
            second_diffs = np.diff(diffs)
            knee_idx = np.argmax(np.abs(second_diffs)) + 1
            threshold = sorted_weights[knee_idx]
            print(f"Knee detection method: threshold = {threshold:.6f} (at position {knee_idx})")
        else:
            # Fallback to median for small graphs
            threshold = np.median(weights)
            print(f"Knee detection fallback (median): threshold = {threshold:.6f}")
    
    elif method == "adaptive":
        # Adaptive method based on graph characteristics
        threshold = get_adaptive_threshold(G, weights)
    
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Print statistics for context
    print(f"Weight statistics:")
    print(f"  Min: {np.min(weights):.6f}")
    print(f"  Max: {np.max(weights):.6f}")
    print(f"  Mean: {np.mean(weights):.6f}")
    print(f"  Median: {np.median(weights):.6f}")
    print(f"  Edges total: {len(weights)}")
    
    # Count how many edges will remain
    remaining_edges = np.sum(weights >= threshold)
    print(f"  Edges >= threshold: {remaining_edges} ({remaining_edges/len(weights)*100:.1f}%)")
    
    return threshold

def get_adaptive_threshold(G, weights):
    """
    Adaptive threshold selection based on graph characteristics.
    """
    import numpy as np
    
    # Separate final decision edges from inter-layer edges
    final_edges = [(u, v, d) for u, v, d in G.edges(data=True) if v.startswith("Final:")]
    inter_edges = [(u, v, d) for u, v, d in G.edges(data=True) if not v.startswith("Final:")]
    
    if final_edges and inter_edges:
        final_weights = [d['weight'] for u, v, d in final_edges]
        inter_weights = [d['weight'] for u, v, d in inter_edges]
        
        # Be more lenient with final decision edges
        final_threshold = np.percentile(final_weights, 10)  # Keep top 90% of final edges
        # Be more selective with inter-layer edges  
        inter_threshold = np.percentile(inter_weights, 85)  # Keep top 15% of inter edges
        
        # Use the minimum to ensure we keep important final paths
        threshold = min(final_threshold, inter_threshold)
        print(f"Adaptive method: final_thresh={final_threshold:.6f}, inter_thresh={inter_threshold:.6f}")
        print(f"Adaptive method: threshold = {threshold:.6f}")
        
    elif final_edges:
        # Only final edges - be conservative
        final_weights = [d['weight'] for u, v, d in final_edges]
        threshold = np.percentile(final_weights, 20)  # Keep top 80%
        print(f"Adaptive method (final only): threshold = {threshold:.6f}")
        
    elif inter_edges:
        # Only inter-layer edges - be selective
        inter_weights = [d['weight'] for u, v, d in inter_edges]
        threshold = np.percentile(inter_weights, 75)  # Keep top 25%
        print(f"Adaptive method (inter only): threshold = {threshold:.6f}")
        
    else:
        # Fallback
        threshold = np.median(weights)
        print(f"Adaptive method (fallback): threshold = {threshold:.6f}")
    
    return threshold

def compare_threshold_methods(G, target_edge_count=None):
    """
    Compare different threshold methods and recommend the best one.
    
    Args:
        G: NetworkX graph
        target_edge_count: Desired number of edges (optional)
    
    Returns:
        dict: Results from all methods
    """
    methods = {
        'conservative': {'method': 'percentile', 'percentile': 85},
        'moderate': {'method': 'std', 'k': 1.0},
        'aggressive': {'method': 'top_k', 'k': 30},
        'adaptive': {'method': 'adaptive'}
    }
    
    results = {}
    
    print("=== COMPARING THRESHOLD METHODS ===")
    for name, params in methods.items():
        print(f"\n--- {name.upper()} METHOD ---")
        threshold = auto_determine_weight_threshold(G, **params)
        edge_count = sum(1 for u, v, d in G.edges(data=True) if d['weight'] >= threshold)
        results[name] = {'threshold': threshold, 'edge_count': edge_count}
    
    # Recommend based on target edge count if provided
    if target_edge_count:
        print(f"\n=== RECOMMENDATION FOR TARGET {target_edge_count} EDGES ===")
        best_method = None
        best_diff = float('inf')
        
        for name, result in results.items():
            diff = abs(result['edge_count'] - target_edge_count)
            if diff < best_diff:
                best_diff = diff
                best_method = name
        
        print(f"Recommended method: {best_method}")
        print(f"  Threshold: {results[best_method]['threshold']:.6f}")
        print(f"  Edge count: {results[best_method]['edge_count']} (target: {target_edge_count})")
        
        return results[best_method]['threshold']
    
    return results

In [ ]:
weight_threshold = auto_determine_weight_threshold(G_reweighted, method="top_k", k=100)

In [ ]:
# # Create and display the cleaned Graphviz visualization with activation values
# print("Creating cleaned Graphviz visualization with activation values...")
# result = create_cleaned_graphviz_with_activations(
#     G_reweighted, text_queries, embeddings, saes_cls, saes_img, 
#     freq_idx_cls, freq_idx_img, layer_concepts_cls, layer_concepts_img, 
#     weight_threshold=0.03
# )

print("Creating cleaned Graphviz visualization with activation values...")
result = create_cleaned_graphviz_with_activations(
    G_reweighted, text_queries, embeddings, saes_cls, saes_img, 
    freq_idx_cls, freq_idx_img, layer_concepts_cls, layer_concepts_img, 
    all_concepts=all_concepts,  # Use pre-computed concepts
    weight_threshold=weight_threshold
)

if result is not None:
    dot_cleaned, G_cleaned_viz, removed_nodes_viz, activation_map, activation_indices_map = result  # MODIFIED: Unpack both maps
    
    try:
        from IPython.display import SVG, display
        svg_data = dot_cleaned.pipe(format="svg")
        display(SVG(svg_data))
        
        # Save to file
        with open("causal_tracing_cleaned_activations.svg", "wb") as f:
            f.write(svg_data)
        print("Saved cleaned graph with activations to causal_tracing_cleaned_activations.svg")
        
    except Exception as e:
        print(f"SVG display failed: {e}")
        print("Trying PNG format instead...")
        
        try:
            png_data = dot_cleaned.pipe(format="png", dpi=200)
            with open("causal_tracing_cleaned_activations.png", "wb") as f:
                f.write(png_data)
            print("Saved as PNG: causal_tracing_cleaned_activations.png")
            
            # Display PNG in matplotlib
            from PIL import Image
            import io
            img = Image.open(io.BytesIO(png_data))
            plt.figure(figsize=(24, 16))
            plt.imshow(img)
            plt.axis('off')
            plt.title("Cleaned Causal Tracing Graph (Showing Activation Values)")
            plt.tight_layout()
            plt.show()
            
        except Exception as e2:
            print(f"PNG generation also failed: {e2}")
            print("Saving as DOT file for external rendering...")
            with open("causal_tracing_cleaned_activations.dot", "w") as f:
                f.write(dot_cleaned.source)
            print("Saved as DOT file: causal_tracing_cleaned_activations.dot")
    
    # Print summary statistics
    print(f"\nCleaned graph summary:")
    print(f"  Nodes: {G_cleaned_viz.number_of_nodes()}")
    print(f"  Edges: {G_cleaned_viz.number_of_edges()}")
    print(f"  Removed dead end nodes: {len(removed_nodes_viz)}")
    
    # Show activation value statistics
    if activation_map:
        activations = list(activation_map.values())
        print(f"\nActivation value statistics:")
        print(f"  Min activation: {min(activations):.4f}")
        print(f"  Max activation: {max(activations):.4f}")
        print(f"  Mean activation: {np.mean(activations):.4f}")
        print(f"  Median activation: {np.median(activations):.4f}")
        
        # Show top activated concepts WITH INDICES
        sorted_activations = sorted(activation_map.items(), key=lambda x: x[1], reverse=True)
        print(f"\nTop 10 activated concepts:")
        for i, (node, activation) in enumerate(sorted_activations[:10], 1):
            layer = node.split(":")[0]
            token_type = node.split(":")[1]
            concept = node.split(":")[-1]
            concept_idx = activation_indices_map.get(node, "N/A")  # NEW: Get the concept index
            print(f"  {i:2d}. {layer} {token_type}: {concept} (Index: {concept_idx}, Act: {activation:.4f})")  # MODIFIED: Include index
    
    # Show remaining nodes by layer
    remaining_by_layer = {}
    for node in G_cleaned_viz.nodes():
        if node.startswith("Final:"):
            layer = "Final"
        else:
            layer = node.split(":")[0]
        remaining_by_layer.setdefault(layer, []).append(node)
    
    print(f"\nRemaining nodes by layer:")
    for layer in sorted(remaining_by_layer.keys(), key=lambda x: int(x[1:]) if x.startswith("L") else 999):
        nodes = remaining_by_layer[layer]
        print(f"  {layer}: {len(nodes)} nodes")
        
else:
    print("Failed to create cleaned visualization - no nodes/edges remaining after cleaning.")